In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    LongType, BooleanType, TimestampType
)
from pyspark.sql.functions import col, from_json

In [0]:
dbutils.widgets.text("catalog_name", "")
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
SECRET_SCOPE = "default2"
personal_conn_string = dbutils.secrets.get(scope=SECRET_SCOPE, key="blech-conn-string-evh")

In [0]:
my_data = {
    "evh_namespace": "evhpl24databricks",
    "evh_name": "blech_evh",
    "evh_conn_string": personal_conn_string
}

target_table = f"{catalog_name}.lechster10_bronze.wiki_evh_bronze"
checkpoint_loc = f"/Volumes/{catalog_name}/lechster10_bronze/lab3_volume/wiki_checkpoints/"

In [0]:
schema = StructType([
    StructField("$schema", StringType(), True),
    StructField("meta", StructType([
        StructField("uri", StringType(), True),
        StructField("request_id", StringType(), True),
        StructField("id", StringType(), True),
        StructField("domain", StringType(), True),
        StructField("stream", StringType(), True),
        StructField("dt", StringType(), True),          
        StructField("topic", StringType(), True),
        StructField("partition", IntegerType(), True),
        StructField("offset", LongType(), True),
    ]), True),
    StructField("id", LongType(), True),
    StructField("type", StringType(), True),             
    StructField("namespace", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("title_url", StringType(), True),
    StructField("comment", StringType(), True),
    StructField("timestamp", LongType(), True),           
    StructField("user", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("notify_url", StringType(), True),
    StructField("minor", BooleanType(), True),
    StructField("patrolled", BooleanType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True),
    ]), True),
    StructField("revision", StructType([
        StructField("old", LongType(), True),
        StructField("new", LongType(), True),
    ]), True),
    StructField("server_url", StringType(), True),
    StructField("server_name", StringType(), True),
    StructField("server_script_path", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("parsedcomment", StringType(), True),
])

In [0]:
df = (spark.readStream
  .format("kafka")
  .option("kafka.bootstrap.servers", f"{my_data['evh_namespace']}.servicebus.windows.net:9093")
  .option("kafka.sasl.mechanism", "PLAIN")
  .option("kafka.security.protocol", "SASL_SSL")          # connection via SSL
  .option("kafka.sasl.jaas.config",
    f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{my_data["evh_conn_string"]}";')
  .option("subscribe", my_data['evh_name'])
  .load()
)


parsed_df = (df
    .withColumn("value", col("value").cast("string"))
    .withColumn("value", from_json(col("value"), schema))    
)


parsed_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_loc) \
    .toTable(target_table)
   

Because it is bronze table, I did not take any data from json (e.g. title) and I did not put it as a single column. I assume that these type of transformations should be in silver layer